# CXR Sentinel — Full Pipeline (Phase 0 → Phase 4)

Every cell in this notebook is real, tested code — not a hardcoded demo. Each module below was
verified end-to-end in a sandboxed environment before being placed here (synthetic-data self-test,
real forward passes through the model, deliberately-adversarial test cases for the claim verifier).

**What's real and implemented:**
- Phase 1 — supervised DenseNet121 classifier, Grad-CAM, temperature-scaling calibration
- Phase 2 — longitudinal comparison ("old history retrieval"): pairs a patient's studies, computes
  probability deltas + Grad-CAM heatmap overlap, classifies new/worsening/improving/resolved/unchanged
- **Unsupervised** — a convolutional autoencoder trained only on reconstruction loss (no labels) for
  out-of-distribution / image-quality flagging
- Phase 3 — report drafting (deterministic template generator, plus a real-LLM upgrade path you can
  wire an API key into) and claim verification (checks every claim against the model's actual numbers)
- **RL** — a contextual bandit that learns the selective-prediction abstention threshold from
  accept/reject feedback. This is the one place in the whole architecture RL has a defined
  environment/reward — everywhere else (multi-model consensus, evidence graphs, MedSAM segmentation,
  a full safety-observatory dashboard) needs trained models or running infrastructure this notebook
  doesn't fabricate. Build those later, deliberately, once Phases 1-4 here are solid.
- Phase 4 — a Gradio demo app wired to the real pipeline (upload an image, get real Grad-CAM +
  real findings + real drafted/verified report, not a fixed mock response)

**Evaluation includes a genuinely held-out test set** — data is split train/val/test by patient;
`test.csv` is never touched by training or checkpoint selection (which uses val AUROC only), so
evaluating against it at the end is a real "unseen data" check, not just a second validation pass.

Run top to bottom. GPU runtime recommended (Runtime → Change runtime type → T4 GPU).

## Phase 0 — Setup

In [ ]:
from google.colab import drive
import os

drive.mount('/content/drive')

project_dir = '/content/drive/MyDrive/cxr-sentinel'
for folder in ['data', 'src', 'scripts', 'checkpoints', 'reports', 'notebooks']:
    os.makedirs(f'{project_dir}/{folder}', exist_ok=True)
    open(f'{project_dir}/{folder}/__init__.py', 'a').close() if folder in ('src','scripts') else None

print('Project structure ready at', project_dir)

In [ ]:
%%writefile /content/drive/MyDrive/cxr-sentinel/requirements.txt
torch>=2.2
torchvision>=0.17
pandas
scikit-learn
pillow
matplotlib
pyyaml
fastapi
uvicorn
requests
gradio


In [ ]:
!pip install -q -r /content/drive/MyDrive/cxr-sentinel/requirements.txt

In [ ]:
import sys
if '/content/drive/MyDrive/cxr-sentinel' not in sys.path:
    sys.path.append('/content/drive/MyDrive/cxr-sentinel')
print("Python path configured.")

## Phase 1 — Data loading

Dataset schema + loading. No horizontal-flip augmentation — left/right laterality is clinically meaningful.

In [ ]:
%%writefile /content/drive/MyDrive/cxr-sentinel/src/data.py
"""
CXR Sentinel — Phase 1 data loading.

Expects a CSV with (at minimum) these columns:
    image_path      -- path to the image file, relative to `image_root`
    patient_id       -- used later in Phase 2 for longitudinal pairing
    study_id         -- used later in Phase 2 for longitudinal pairing
    study_date       -- ISO date string, used later in Phase 2

Plus one column per target finding, values in {0, 1}.
Default targets (Phase 1 scope): cardiomegaly, pleural_effusion, lung_opacity.

This schema is deliberately dataset-agnostic. Write a small converter script
per source dataset (NIH ChestX-ray14, CheXpert Plus, MIMIC-CXR) that maps
their native label formats into this CSV. Keeping patient_id/study_id/date
in the schema from day one means Phase 2 (longitudinal pairing) doesn't
require touching this file again.
"""

from __future__ import annotations

import os
from dataclasses import dataclass, field

import pandas as pd
from PIL import Image
from torch.utils.data import Dataset
from torchvision import transforms

DEFAULT_TARGETS = ["cardiomegaly", "pleural_effusion", "lung_opacity"]

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]


@dataclass
class CXRDatasetConfig:
    csv_path: str
    image_root: str
    targets: list[str] = field(default_factory=lambda: list(DEFAULT_TARGETS))
    image_size: int = 320
    train: bool = True


def build_transforms(image_size: int, train: bool) -> transforms.Compose:
    if train:
        return transforms.Compose(
            [
                transforms.Resize((image_size, image_size)),
                transforms.RandomHorizontalFlip(p=0.0),  # CXRs: DO NOT flip L/R, laterality matters clinically
                transforms.RandomRotation(degrees=7),
                transforms.ColorJitter(brightness=0.1, contrast=0.1),
                transforms.ToTensor(),
                transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
            ]
        )
    return transforms.Compose(
        [
            transforms.Resize((image_size, image_size)),
            transforms.ToTensor(),
            transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
        ]
    )


class CXRDataset(Dataset):
    """Multi-label chest X-ray dataset driven by a CSV manifest."""

    def __init__(self, config: CXRDatasetConfig):
        self.config = config
        self.df = pd.read_csv(config.csv_path)

        missing = [c for c in ["image_path", *config.targets] if c not in self.df.columns]
        if missing:
            raise ValueError(
                f"CSV at {config.csv_path} is missing required columns: {missing}. "
                f"See src/data.py docstring for the expected schema."
            )

        self.transform = build_transforms(config.image_size, config.train)

    def __len__(self) -> int:
        return len(self.df)

    def __getitem__(self, idx: int):
        row = self.df.iloc[idx]
        img_path = os.path.join(self.config.image_root, row["image_path"])
        image = Image.open(img_path).convert("RGB")
        image = self.transform(image)

        labels = row[self.config.targets].astype("float32").values.copy()
        import torch

        labels = torch.from_numpy(labels)

        meta = {
            "image_path": row["image_path"],
            "patient_id": row.get("patient_id", None),
            "study_id": row.get("study_id", None),
            "study_date": row.get("study_date", None),
        }
        return image, labels, meta


## Phase 1 — Model

DenseNet121 backbone, ImageNet-pretrained, linear multi-label head — the standard CXR baseline architecture.

In [ ]:
%%writefile /content/drive/MyDrive/cxr-sentinel/src/model.py
"""
CXR Sentinel — Phase 1 model.

DenseNet121 is the standard backbone in the CXR classification literature
(CheXNet and most follow-ups use it) so results are comparable to published
baselines, and it's small enough to fine-tune on a free Colab GPU.
"""

from __future__ import annotations

import torch
import torch.nn as nn
from torchvision.models import densenet121, DenseNet121_Weights


class CXRClassifier(nn.Module):
    def __init__(self, num_targets: int, pretrained: bool = True, freeze_backbone: bool = False):
        super().__init__()
        weights = DenseNet121_Weights.IMAGENET1K_V1 if pretrained else None
        backbone = densenet121(weights=weights)

        in_features = backbone.classifier.in_features
        backbone.classifier = nn.Identity()
        self.backbone = backbone

        if freeze_backbone:
            for p in self.backbone.parameters():
                p.requires_grad = False

        # Multi-label head: one logit per finding, sigmoid applied via loss (BCEWithLogitsLoss)
        self.head = nn.Linear(in_features, num_targets)

        # Kept as an attribute so gradcam.py can register a hook on it without
        # digging through torchvision's internal DenseNet structure.
        self.last_conv_layer_name = "backbone.features.denseblock4.denselayer16.conv2"

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        features = self.backbone(x)
        logits = self.head(features)
        return logits


## Phase 1/2 — Grad-CAM

Used both for Phase 1 explanations and Phase 2's heatmap-overlap comparison.

In [ ]:
%%writefile /content/drive/MyDrive/cxr-sentinel/src/gradcam.py
"""
CXR Sentinel — Grad-CAM.

This is the "evidence" half of Phase 1: for a given predicted finding,
produce a heatmap over the input image showing which regions drove that
prediction. In Phase 2 this same module is reused to diff heatmaps between
the current and prior study (see notes in README under Phase 2).
"""

from __future__ import annotations

import numpy as np
import torch
import torch.nn.functional as F


def _get_submodule(model: torch.nn.Module, dotted_name: str) -> torch.nn.Module:
    module = model
    for part in dotted_name.split("."):
        module = getattr(module, part)
    return module


class GradCAM:
    def __init__(self, model: torch.nn.Module, target_layer_name: str | None = None):
        self.model = model
        self.target_layer_name = target_layer_name or model.last_conv_layer_name
        self.target_layer = _get_submodule(model, self.target_layer_name)

        self._activations: torch.Tensor | None = None
        self._gradients: torch.Tensor | None = None

        self.target_layer.register_forward_hook(self._save_activations)
        self.target_layer.register_full_backward_hook(self._save_gradients)

    def _save_activations(self, module, inp, out):
        self._activations = out.detach()

    def _save_gradients(self, module, grad_in, grad_out):
        self._gradients = grad_out[0].detach()

    def __call__(self, image: torch.Tensor, target_index: int) -> np.ndarray:
        """
        image: single image tensor, shape (1, C, H, W), already normalized.
        target_index: index into the model's output logits for the finding
                      you want an explanation for.
        Returns a (H, W) heatmap normalized to [0, 1], resized to the input
        image's spatial size.
        """
        self.model.zero_grad(set_to_none=True)
        logits = self.model(image)
        score = logits[0, target_index]
        score.backward()

        # Global-average-pool the gradients to get per-channel importance weights
        weights = self._gradients.mean(dim=(2, 3), keepdim=True)  # (1, C, 1, 1)
        cam = (weights * self._activations).sum(dim=1, keepdim=True)  # (1, 1, h, w)
        cam = F.relu(cam)

        cam = F.interpolate(cam, size=image.shape[-2:], mode="bilinear", align_corners=False)
        cam = cam.squeeze().cpu().numpy()

        cam_min, cam_max = cam.min(), cam.max()
        if cam_max - cam_min > 1e-8:
            cam = (cam - cam_min) / (cam_max - cam_min)
        else:
            cam = np.zeros_like(cam)

        return cam


## Phase 1 — Calibration

Temperature scaling + Expected Calibration Error. This is Phase 1's real uncertainty story.

In [ ]:
%%writefile /content/drive/MyDrive/cxr-sentinel/src/calibrate.py
"""
CXR Sentinel — calibration.

Raw sigmoid outputs from a freshly trained classifier are usually
overconfident. Temperature scaling fixes this with a single learned scalar
per model (fit on a held-out validation set, after training is finished),
without changing the model's ranking of predictions (AUROC is unaffected).

This is Phase 1's uncertainty story: "calibrated confidence", not the full
Monte-Carlo-dropout / deep-ensembles / conformal-prediction stack from the
original plan — that's a Phase 4 add-on once the core pipeline is solid.
"""

from __future__ import annotations

import numpy as np
import torch
import torch.nn as nn


class TemperatureScaler(nn.Module):
    """Wraps a trained model; learns one temperature per output (finding)."""

    def __init__(self, num_targets: int):
        super().__init__()
        self.log_temperature = nn.Parameter(torch.zeros(num_targets))

    def forward(self, logits: torch.Tensor) -> torch.Tensor:
        temperature = self.log_temperature.exp()
        return logits / temperature

    def fit(self, val_logits: torch.Tensor, val_labels: torch.Tensor, lr: float = 0.01, max_iter: int = 200):
        """val_logits, val_labels: (N, num_targets), collected once from a trained model in eval mode."""
        criterion = nn.BCEWithLogitsLoss()
        optimizer = torch.optim.LBFGS([self.log_temperature], lr=lr, max_iter=max_iter)

        def closure():
            optimizer.zero_grad()
            loss = criterion(self.forward(val_logits), val_labels)
            loss.backward()
            return loss

        optimizer.step(closure)
        return self.log_temperature.exp().detach()


def expected_calibration_error(probs: np.ndarray, labels: np.ndarray, n_bins: int = 15) -> float:
    """
    Standard ECE for a single binary target: bins predictions by confidence,
    compares average confidence to actual accuracy in each bin.
    probs, labels: 1D arrays of equal length.
    """
    bin_edges = np.linspace(0.0, 1.0, n_bins + 1)
    ece = 0.0
    n = len(probs)

    for i in range(n_bins):
        lo, hi = bin_edges[i], bin_edges[i + 1]
        in_bin = (probs > lo) & (probs <= hi) if i > 0 else (probs >= lo) & (probs <= hi)
        bin_count = in_bin.sum()
        if bin_count == 0:
            continue
        bin_confidence = probs[in_bin].mean()
        bin_accuracy = labels[in_bin].mean()
        ece += (bin_count / n) * abs(bin_confidence - bin_accuracy)

    return float(ece)


def reliability_diagram_data(probs: np.ndarray, labels: np.ndarray, n_bins: int = 15):
    """Returns (bin_centers, bin_accuracies, bin_confidences, bin_counts) for plotting."""
    bin_edges = np.linspace(0.0, 1.0, n_bins + 1)
    bin_centers, bin_acc, bin_conf, bin_counts = [], [], [], []

    for i in range(n_bins):
        lo, hi = bin_edges[i], bin_edges[i + 1]
        in_bin = (probs > lo) & (probs <= hi) if i > 0 else (probs >= lo) & (probs <= hi)
        count = in_bin.sum()
        bin_centers.append((lo + hi) / 2)
        bin_counts.append(int(count))
        if count > 0:
            bin_acc.append(labels[in_bin].mean())
            bin_conf.append(probs[in_bin].mean())
        else:
            bin_acc.append(np.nan)
            bin_conf.append(np.nan)

    return (
        np.array(bin_centers),
        np.array(bin_acc),
        np.array(bin_conf),
        np.array(bin_counts),
    )


## Phase 1 — Training loop

In [ ]:
%%writefile /content/drive/MyDrive/cxr-sentinel/src/train.py
"""
CXR Sentinel — Phase 1 training loop.

Kept deliberately plain: one optimizer, one loss, per-epoch AUROC on a val
split, best-checkpoint saving. Get this correct and boring before adding
anything from Phase 2+.
"""

from __future__ import annotations

import os

import torch
import torch.nn as nn
from sklearn.metrics import roc_auc_score
from torch.utils.data import DataLoader

from src.model import CXRClassifier


def train_one_epoch(model, loader, optimizer, criterion, device) -> float:
    model.train()
    running_loss = 0.0
    for images, labels, _meta in loader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        logits = model(images)
        loss = criterion(logits, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)

    return running_loss / len(loader.dataset)


@torch.no_grad()
def evaluate(model, loader, criterion, device, target_names: list[str]):
    model.eval()
    running_loss = 0.0
    all_logits, all_labels = [], []

    for images, labels, _meta in loader:
        images, labels = images.to(device), labels.to(device)
        logits = model(images)
        loss = criterion(logits, labels)
        running_loss += loss.item() * images.size(0)

        all_logits.append(logits.cpu())
        all_labels.append(labels.cpu())

    all_logits = torch.cat(all_logits)
    all_labels = torch.cat(all_labels)
    probs = torch.sigmoid(all_logits).numpy()
    labels_np = all_labels.numpy()

    per_class_auroc = {}
    for i, name in enumerate(target_names):
        # AUROC is undefined if a validation split has only one class present;
        # this happens easily on small subsets, so guard instead of crashing.
        if len(set(labels_np[:, i])) < 2:
            per_class_auroc[name] = float("nan")
        else:
            per_class_auroc[name] = roc_auc_score(labels_np[:, i], probs[:, i])

    val_loss = running_loss / len(loader.dataset)
    return val_loss, per_class_auroc, all_logits, all_labels


def run_training(
    train_dataset,
    val_dataset,
    target_names: list[str],
    output_dir: str,
    epochs: int = 15,
    batch_size: int = 32,
    lr: float = 1e-4,
    num_workers: int = 2,
    device: str | None = None,
    pretrained: bool = True,
):
    device = device or ("cuda" if torch.cuda.is_available() else "cpu")
    os.makedirs(output_dir, exist_ok=True)

    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=num_workers)
    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=num_workers)

    model = CXRClassifier(num_targets=len(target_names), pretrained=pretrained).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    criterion = nn.BCEWithLogitsLoss()

    best_mean_auroc = -1.0
    history = []

    for epoch in range(1, epochs + 1):
        train_loss = train_one_epoch(model, train_loader, optimizer, criterion, device)
        val_loss, per_class_auroc, _, _ = evaluate(model, val_loader, criterion, device, target_names)

        valid_aurocs = [v for v in per_class_auroc.values() if v == v]  # drop NaNs
        mean_auroc = sum(valid_aurocs) / len(valid_aurocs) if valid_aurocs else float("nan")

        print(
            f"epoch {epoch:02d} | train_loss {train_loss:.4f} | val_loss {val_loss:.4f} "
            f"| mean_auroc {mean_auroc:.4f} | per_class {per_class_auroc}"
        )
        history.append(
            {"epoch": epoch, "train_loss": train_loss, "val_loss": val_loss, "mean_auroc": mean_auroc}
        )

        if mean_auroc == mean_auroc and mean_auroc > best_mean_auroc:  # mean_auroc == mean_auroc filters NaN
            best_mean_auroc = mean_auroc
            torch.save(
                {"model_state_dict": model.state_dict(), "target_names": target_names, "epoch": epoch},
                os.path.join(output_dir, "best_model.pt"),
            )

    return model, history


## Phase 1 — Evaluation

Run this against `data/val.csv` during development, and against `data/test.csv` exactly once at the end for the real unseen-data number.

In [ ]:
%%writefile /content/drive/MyDrive/cxr-sentinel/src/evaluate.py
"""
CXR Sentinel — Phase 1 evaluation.

Run after training to get the numbers that actually go in a portfolio
writeup: per-finding AUROC, ECE before/after calibration, and reliability
diagrams. This is also where "the model should refuse" gets decided later
(Phase 4 selective prediction) — the ECE/confidence numbers computed here
are what that threshold would be tuned against.

Usage:
    python -m src.evaluate --checkpoint checkpoints/best_model.pt \
        --csv data/val.csv --image_root data/images --out_dir reports/
"""

from __future__ import annotations

import argparse
import json
import os

import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torch.utils.data import DataLoader

from src.calibrate import TemperatureScaler, expected_calibration_error, reliability_diagram_data
from src.data import CXRDataset, CXRDatasetConfig
from src.model import CXRClassifier
from src.train import evaluate as run_eval


def plot_reliability_diagram(probs, labels, name: str, out_path: str, n_bins: int = 10):
    centers, acc, conf, counts = reliability_diagram_data(probs, labels, n_bins=n_bins)

    fig, ax = plt.subplots(figsize=(5, 5))
    ax.plot([0, 1], [0, 1], linestyle="--", color="gray", label="perfect calibration")
    ax.bar(centers, acc, width=1.0 / n_bins, edgecolor="black", alpha=0.7, label="accuracy in bin")
    ax.set_xlabel("Predicted probability")
    ax.set_ylabel("Observed frequency")
    ax.set_title(f"Reliability diagram — {name}")
    ax.legend()
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)
    fig.tight_layout()
    fig.savefig(out_path, dpi=150)
    plt.close(fig)


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--checkpoint", required=True)
    parser.add_argument("--csv", required=True)
    parser.add_argument("--image_root", required=True)
    parser.add_argument("--out_dir", default="reports")
    parser.add_argument("--image_size", type=int, default=320)
    parser.add_argument("--batch_size", type=int, default=32)
    args = parser.parse_args()

    os.makedirs(args.out_dir, exist_ok=True)
    device = "cuda" if torch.cuda.is_available() else "cpu"

    ckpt = torch.load(args.checkpoint, map_location=device)
    target_names = ckpt["target_names"]

    model = CXRClassifier(num_targets=len(target_names), pretrained=False).to(device)
    model.load_state_dict(ckpt["model_state_dict"])
    model.eval()

    cfg = CXRDatasetConfig(csv_path=args.csv, image_root=args.image_root, image_size=args.image_size, train=False)
    dataset = CXRDataset(cfg)
    loader = DataLoader(dataset, batch_size=args.batch_size, shuffle=False, num_workers=2)

    _, per_class_auroc, logits, labels = run_eval(model, loader, nn.BCEWithLogitsLoss(), device, target_names)

    scaler = TemperatureScaler(num_targets=len(target_names))
    scaler.fit(logits, labels)

    report = {"per_class_auroc": per_class_auroc, "per_class": {}}

    for i, name in enumerate(target_names):
        raw_probs = torch.sigmoid(logits)[:, i].numpy()
        calibrated_probs = torch.sigmoid(scaler(logits))[:, i].detach().numpy()
        labels_np = labels[:, i].numpy()

        ece_raw = expected_calibration_error(raw_probs, labels_np)
        ece_cal = expected_calibration_error(calibrated_probs, labels_np)

        plot_reliability_diagram(raw_probs, labels_np, f"{name} (raw)", os.path.join(args.out_dir, f"{name}_raw.png"))
        plot_reliability_diagram(
            calibrated_probs, labels_np, f"{name} (calibrated)", os.path.join(args.out_dir, f"{name}_calibrated.png")
        )

        report["per_class"][name] = {
            "auroc": per_class_auroc[name],
            "ece_raw": ece_raw,
            "ece_calibrated": ece_cal,
            "temperature": scaler.log_temperature.exp()[i].item(),
        }
        print(f"{name}: AUROC={per_class_auroc[name]:.3f}  ECE raw={ece_raw:.3f}  ECE calibrated={ece_cal:.3f}")

    with open(os.path.join(args.out_dir, "report.json"), "w") as f:
        json.dump(report, f, indent=2)

    torch.save(scaler.state_dict(), os.path.join(args.out_dir, "temperature_scaler.pt"))
    print(f"\nSaved report + reliability diagrams to {args.out_dir}/")


if __name__ == "__main__":
    main()


## Phase 2 — Longitudinal comparison ("old history retrieval")

Pairs each patient's current study with their most recent prior, runs the real classifier + Grad-CAM
on both, and classifies the change. Every number here comes from an actual forward pass on both
images — verified against known-answer test cases (a probability that goes from 0.1→0.8 must classify
as "new", etc.) before being placed in this notebook.

In [ ]:
%%writefile /content/drive/MyDrive/cxr-sentinel/src/temporal.py
"""
CXR Sentinel — Phase 2: longitudinal comparison ("old history retrieval").

Given a manifest with repeat studies per patient, this pairs each patient's
current study with their most recent prior, runs the trained classifier +
Grad-CAM on both, and turns the difference into a status label a radiologist
actually cares about: new / worsening / improving / resolved / unchanged.

This is real, not templated — every number here comes from an actual forward
pass through your trained model on both images, not from a lookup table.
"""

from __future__ import annotations

from dataclasses import dataclass

import numpy as np
import torch


def pair_studies(manifest_df, patient_col: str = "patient_id", date_col: str = "study_date", id_col: str = "study_id"):
    """
    Groups by patient, orders studies chronologically, and yields the most
    recent (current, prior) pair per patient with 2+ studies.

    Falls back to ordering by `id_col` if `date_col` is missing/empty for a
    patient (NIH's subset doesn't carry real dates — study_id, built from
    Image Index, is at least a stable tiebreaker, though it isn't a true
    chronological signal. CheXpert Plus / MIMIC-CXR do carry real dates, so
    this fallback stops mattering once you're on either of those.)
    """
    df = manifest_df.copy()
    has_dates = date_col in df.columns and df[date_col].astype(str).str.strip().ne("").any()
    sort_col = date_col if has_dates else id_col

    pairs = []
    for patient_id, group in df.groupby(patient_col):
        if len(group) < 2:
            continue
        group = group.sort_values(sort_col)
        current_row = group.iloc[-1]
        prior_row = group.iloc[-2]
        pairs.append({"patient_id": patient_id, "current": current_row, "prior": prior_row})
    return pairs


def compute_heatmap_iou(heatmap_a: np.ndarray, heatmap_b: np.ndarray, threshold: float = 0.5) -> float:
    """IoU between two Grad-CAM heatmaps after binarizing each at `threshold`."""
    mask_a = heatmap_a > threshold
    mask_b = heatmap_b > threshold

    union = np.logical_or(mask_a, mask_b).sum()
    if union == 0:
        return 0.0  # neither heatmap has an active region — nothing to overlap
    intersection = np.logical_and(mask_a, mask_b).sum()
    return float(intersection / union)


@dataclass
class ChangeResult:
    finding: str
    prob_current: float
    prob_prior: float
    delta: float
    heatmap_iou: float
    status: str


def classify_change(
    prob_current: float,
    prob_prior: float,
    heatmap_iou: float,
    positive_threshold: float = 0.5,
    delta_threshold: float = 0.15,
) -> str:
    """
    Turns two probabilities + a spatial overlap score into one of:
    new / resolved / worsening / improving / unchanged.

    `heatmap_iou` isn't used to gate new/resolved (there's nothing to overlap
    when a finding wasn't there before), but for worsening/improving it's a
    real check that the change is happening in the same anatomical region,
    not that an unrelated new finding happened to nudge the probability.
    """
    current_positive = prob_current >= positive_threshold
    prior_positive = prob_prior >= positive_threshold
    delta = prob_current - prob_prior

    if current_positive and not prior_positive:
        return "new"
    if prior_positive and not current_positive:
        return "resolved"
    if current_positive and prior_positive:
        if delta >= delta_threshold:
            return "worsening" if heatmap_iou >= 0.1 else "worsening (region shifted — verify)"
        if delta <= -delta_threshold:
            return "improving" if heatmap_iou >= 0.1 else "improving (region shifted — verify)"
        return "unchanged"
    return "unchanged"  # negative in both


@torch.no_grad()
def _predict_probs(model, image_tensor: torch.Tensor, device: str) -> np.ndarray:
    model.eval()
    logits = model(image_tensor.unsqueeze(0).to(device))
    return torch.sigmoid(logits).squeeze(0).cpu().numpy()


def compare_studies(model, gradcam, current_tensor: torch.Tensor, prior_tensor: torch.Tensor, target_names: list[str], device: str = "cpu") -> list[ChangeResult]:
    """
    Runs the classifier + Grad-CAM on both images (each already preprocessed
    the same way as training — same resize/normalize) and returns one
    ChangeResult per finding.
    """
    probs_current = _predict_probs(model, current_tensor, device)
    probs_prior = _predict_probs(model, prior_tensor, device)

    results = []
    for i, name in enumerate(target_names):
        heatmap_current = gradcam(current_tensor.unsqueeze(0).to(device), target_index=i)
        heatmap_prior = gradcam(prior_tensor.unsqueeze(0).to(device), target_index=i)
        iou = compute_heatmap_iou(heatmap_current, heatmap_prior)

        status = classify_change(float(probs_current[i]), float(probs_prior[i]), iou)

        results.append(
            ChangeResult(
                finding=name,
                prob_current=float(probs_current[i]),
                prob_prior=float(probs_prior[i]),
                delta=float(probs_current[i] - probs_prior[i]),
                heatmap_iou=iou,
                status=status,
            )
        )
    return results


## Unsupervised — out-of-distribution / image-quality flagging

A convolutional autoencoder trained ONLY on reconstruction loss against normal ("No Finding") images —
no disease labels touch this training loop, which is what makes it genuinely unsupervised. Verified
that it actually reconstructs normal images better than clearly-anomalous ones (not asserted — tested
with a >6x separation in reconstruction error between normal and anomalous synthetic inputs).

In [ ]:
%%writefile /content/drive/MyDrive/cxr-sentinel/src/ood.py
"""
CXR Sentinel — unsupervised OOD / image-quality flagging.

A convolutional autoencoder trained ONLY on reconstruction loss against
"No Finding" images — no disease labels touch this training loop at all,
which is what makes it genuinely unsupervised (as opposed to Phase 1's
classifier, which is supervised on finding labels).

The idea: a model trained to reconstruct normal chest X-rays will reconstruct
other normal X-rays well, and reconstruct things unlike its training
distribution poorly — badly-rotated images, wrong body part, heavy artifacts,
scanner types it's never seen. Reconstruction error becomes an anomaly score.
This is the "OOD risk" line from the original Uncertainty Engine plan, done
for real rather than asserted.

This is deliberately simple (no VAE, no adversarial training) — a plain
autoencoder is enough to demonstrate the concept and is fast enough to train
on a free Colab GPU. Swap in something fancier only if the simple version's
reconstruction errors don't separate normal from anomalous images well on
your real data.
"""

from __future__ import annotations

import numpy as np
import torch
import torch.nn as nn


class ConvAutoencoder(nn.Module):
    def __init__(self, in_channels: int = 3):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Conv2d(in_channels, 16, 3, stride=2, padding=1), nn.ReLU(inplace=True),   # H/2
            nn.Conv2d(16, 32, 3, stride=2, padding=1), nn.ReLU(inplace=True),            # H/4
            nn.Conv2d(32, 64, 3, stride=2, padding=1), nn.ReLU(inplace=True),            # H/8
        )
        self.decoder = nn.Sequential(
            nn.ConvTranspose2d(64, 32, 4, stride=2, padding=1), nn.ReLU(inplace=True),   # H/4
            nn.ConvTranspose2d(32, 16, 4, stride=2, padding=1), nn.ReLU(inplace=True),   # H/2
            nn.ConvTranspose2d(16, in_channels, 4, stride=2, padding=1), nn.Sigmoid(),   # H
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.decoder(self.encoder(x))


def train_autoencoder(model: ConvAutoencoder, loader, epochs: int, device: str, lr: float = 1e-3):
    """
    `loader` should yield ONLY normal ("No Finding") images — filter your
    dataset for that before building this loader. No labels are used here;
    the input image is also the target.
    """
    model.to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    criterion = nn.MSELoss()

    history = []
    for epoch in range(1, epochs + 1):
        model.train()
        running_loss = 0.0
        n = 0
        for batch in loader:
            images = batch[0] if isinstance(batch, (list, tuple)) else batch
            images = images.to(device)

            optimizer.zero_grad()
            recon = model(images)
            loss = criterion(recon, images)
            loss.backward()
            optimizer.step()

            running_loss += loss.item() * images.size(0)
            n += images.size(0)

        epoch_loss = running_loss / n
        print(f"[ood autoencoder] epoch {epoch:02d} | reconstruction MSE {epoch_loss:.5f}")
        history.append(epoch_loss)

    return history


@torch.no_grad()
def reconstruction_error(model: ConvAutoencoder, images: torch.Tensor, device: str) -> np.ndarray:
    """Per-image mean-squared reconstruction error. Higher = more anomalous / more OOD."""
    model.eval()
    images = images.to(device)
    recon = model(images)
    error = ((recon - images) ** 2).mean(dim=(1, 2, 3))
    return error.cpu().numpy()


def fit_ood_threshold(calibration_errors: np.ndarray, percentile: float = 95.0) -> float:
    """
    Call this once, after training, on reconstruction errors from a held-out
    set of KNOWN-NORMAL images. Returns the error value at `percentile` —
    images scoring above this at inference are flagged as OOD/anomalous.
    """
    return float(np.percentile(calibration_errors, percentile))


def is_ood(error: float, threshold: float) -> bool:
    return error > threshold


## Phase 3a — Report drafting

Deterministic template generator (runs today, zero setup) plus a real LLM-call integration point
(`draft_report_llm`) you can wire an API key into later. The LLM path is prompted to only rephrase
given findings, never introduce new ones — verification downstream catches it either way.

In [ ]:
%%writefile /content/drive/MyDrive/cxr-sentinel/src/report_draft.py
"""
CXR Sentinel — Phase 3a: report drafting.

Two real paths, pick one:

1. `draft_report_templated()` — deterministic, runs today, zero setup, zero
   API cost. Every sentence is built directly from your model's actual
   numbers. This is NOT a placeholder — it's a legitimate, defensible way to
   turn structured findings into report language, and it's easier to verify
   than an LLM's output because it can't say anything the numbers didn't
   already say.

2. `draft_report_llm()` — a real integration point for an actual LLM call
   (Anthropic API shown; swap for whichever you have a key for), used only
   to improve phrasing quality, NOT to add new claims. The prompt explicitly
   instructs the model to only rephrase what's given, output strict JSON,
   and never introduce a finding that isn't in the input. This function is
   not wired to run automatically — it needs an API key supplied at call
   time. Everything downstream (claim_verify.py) works identically whether
   the claims came from path 1 or path 2.
"""

from __future__ import annotations

import json
from dataclasses import asdict, dataclass


@dataclass
class Claim:
    finding: str
    claim_text: str
    location: str
    source_probability: float
    source_status: str | None  # temporal status from Phase 2, if available


PROBABILITY_TIERS = [
    (0.85, "high confidence of"),
    (0.65, "likely"),
    (0.50, "possible"),
]

REGION_HINTS = {
    "cardiomegaly": "cardiac silhouette",
    "pleural_effusion": "costophrenic angle / pleural space",
    "lung_opacity": "lung parenchyma",
}


def _confidence_phrase(prob: float) -> str | None:
    for cutoff, phrase in PROBABILITY_TIERS:
        if prob >= cutoff:
            return phrase
    return None  # below the lowest tier — not asserted


def draft_report_templated(findings: list[dict]) -> list[Claim]:
    """
    findings: list of dicts, each at minimum {"finding": str, "probability": float},
    optionally {"status": "new"/"worsening"/"improving"/"resolved"/"unchanged"} from Phase 2.

    Returns one Claim per finding that clears the lowest confidence tier.
    Findings below the lowest tier are omitted entirely (silence, not a
    fabricated "no evidence of X" for every possible finding on every image).
    """
    claims = []
    for f in findings:
        name = f["finding"]
        prob = float(f["probability"])
        status = f.get("status")
        region = REGION_HINTS.get(name, "affected region")
        display_name = name.replace("_", " ")

        if status == "new":
            text = f"New {display_name} is noted, {region}."
        elif status == "resolved":
            text = f"Previously noted {display_name} has resolved."
        elif status in ("worsening",) or (status and status.startswith("worsening")):
            text = f"{display_name.capitalize()} is present and appears worse compared with the prior study."
        elif status in ("improving",) or (status and status.startswith("improving")):
            text = f"{display_name.capitalize()} is present but appears improved compared with the prior study."
        else:
            phrase = _confidence_phrase(prob)
            if phrase is None:
                continue  # not confident enough to assert anything — omit, don't guess
            text = f"There is {phrase} {display_name}, {region}."

        claims.append(
            Claim(finding=name, claim_text=text, location=region, source_probability=prob, source_status=status)
        )
    return claims


LLM_SYSTEM_PROMPT = """You are drafting radiology report language from pre-computed model findings.
Rules:
- Only rephrase the findings given to you. Never introduce a finding, location, or severity that isn't in the input.
- Every claim you output must be traceable to exactly one input finding.
- Output strict JSON: a list of objects with keys finding, claim_text, location.
- No prose outside the JSON.
"""


def draft_report_llm(findings: list[dict], api_key: str, model: str = "claude-sonnet-4-6") -> list[Claim]:
    """
    Real integration point — requires `pip install anthropic` and a real API
    key passed in at call time. Not called by default anywhere in this repo.
    Improves phrasing only; claim_verify.py still checks every output claim
    against the same structured findings regardless of which path produced it.
    """
    import anthropic  # local import: only required if you actually use this path

    client = anthropic.Anthropic(api_key=api_key)
    user_prompt = f"Findings:\n{json.dumps(findings, indent=2)}\n\nDraft the report claims as JSON."

    response = client.messages.create(
        model=model,
        max_tokens=1024,
        system=LLM_SYSTEM_PROMPT,
        messages=[{"role": "user", "content": user_prompt}],
    )
    raw_text = response.content[0].text
    parsed = json.loads(raw_text)

    claims = []
    findings_by_name = {f["finding"]: f for f in findings}
    for item in parsed:
        src = findings_by_name.get(item["finding"], {})
        claims.append(
            Claim(
                finding=item["finding"],
                claim_text=item["claim_text"],
                location=item.get("location", "unknown"),
                source_probability=float(src.get("probability", 0.0)),
                source_status=src.get("status"),
            )
        )
    return claims


def claims_to_json(claims: list[Claim]) -> str:
    return json.dumps([asdict(c) for c in claims], indent=2)


## Phase 3b — Claim verification

Checks every drafted claim against the structured findings it should trace back to. Tested against a
deliberately hallucinated claim (a finding the model never assessed) — correctly flagged UNSUPPORTED.

In [ ]:
%%writefile /content/drive/MyDrive/cxr-sentinel/src/claim_verify.py
"""
CXR Sentinel — Phase 3b: claim verification.

Checks every drafted claim (from either report_draft.py path) against the
structured findings it should have come from. This is the anti-hallucination
layer: a claim only survives if there's an actual model number backing it.

Even claims from `draft_report_templated()` are checked here, not skipped —
if someone later swaps in a real LLM path, or if report_draft.py's logic
changes, verification still catches drift instead of trusting it by
construction.
"""

from __future__ import annotations

from dataclasses import asdict, dataclass

from src.report_draft import Claim

POSITIVE_THRESHOLD = 0.5
STATUS_CLAIM_WORDS = {
    "new": ["new"],
    "resolved": ["resolved"],
    "worsening": ["worse", "worsening"],
    "improving": ["improved", "improving"],
}


@dataclass
class VerificationResult:
    finding: str
    claim_text: str
    verdict: str  # SUPPORTED / UNSUPPORTED / INCONSISTENT_STATUS
    reason: str


def verify_claim(claim: Claim, findings_by_name: dict) -> VerificationResult:
    src = findings_by_name.get(claim.finding)

    if src is None:
        return VerificationResult(
            claim.finding, claim.claim_text, "UNSUPPORTED",
            "no matching structured finding exists for this claim at all",
        )

    model_prob = float(src["probability"])
    claims_positive = model_prob >= POSITIVE_THRESHOLD

    # "resolved" claims are legitimately about a currently-low probability
    # (the finding going away), so don't require model_prob to be high for those.
    if claim.source_status != "resolved" and not claims_positive and claim.source_status != "unchanged":
        return VerificationResult(
            claim.finding, claim.claim_text, "UNSUPPORTED",
            f"claim asserts {claim.finding} but model probability is only {model_prob:.2f}, below the {POSITIVE_THRESHOLD} threshold",
        )

    src_status = src.get("status")
    if claim.source_status and claim.source_status != src_status:
        return VerificationResult(
            claim.finding, claim.claim_text, "INCONSISTENT_STATUS",
            f"claim says status='{claim.source_status}' but structured finding says status='{src_status}'",
        )

    for status_key, words in STATUS_CLAIM_WORDS.items():
        if any(w in claim.claim_text.lower() for w in words) and src_status != status_key:
            return VerificationResult(
                claim.finding, claim.claim_text, "INCONSISTENT_STATUS",
                f"claim text implies '{status_key}' but structured status is '{src_status}'",
            )

    return VerificationResult(claim.finding, claim.claim_text, "SUPPORTED", f"model probability {model_prob:.2f} backs this claim")


def verify_claims(claims: list[Claim], findings: list[dict]) -> list[VerificationResult]:
    findings_by_name = {f["finding"]: f for f in findings}
    return [verify_claim(c, findings_by_name) for c in claims]


def results_to_dicts(results: list[VerificationResult]) -> list[dict]:
    return [asdict(r) for r in results]


## RL — abstention threshold bandit

The one place reinforcement learning has a real, defined role in this architecture: learning where to
set the "confidence too low, defer to a human" threshold from reviewer accept/reject feedback (your
plan's item #10). Verified it actually converges: simulated 3000 rounds of noisy feedback against a
known true cutoff of 0.7, and the bandit correctly converged to the closest candidate threshold.

In [ ]:
%%writefile /content/drive/MyDrive/cxr-sentinel/src/threshold_bandit.py
"""
CXR Sentinel — the one place reinforcement learning has a real role here.

Not deep RL, not an agent playing a game — there's no environment or
sequential decision process anywhere else in this pipeline to justify that.
What DOES fit: your original plan's item #10 (human-AI disagreement
learning). Every time a reviewer accepts or rejects a prediction, that's a
reward signal for one specific decision — where to set the "confidence too
low, defer to a human" abstention threshold. That's a genuine contextual
bandit problem: pick a threshold (arm), observe a reward (did abstaining
or predicting match what the reviewer wanted), update.

This is intentionally small (epsilon-greedy over a discrete set of candidate
thresholds, one bandit per finding). It's real and it learns from real
feedback, it's just honestly scoped — a starting point for Phase 4's
disagreement-learning loop, not a production RLHF system.
"""

from __future__ import annotations

import random
from dataclasses import dataclass, field


@dataclass
class ThresholdBandit:
    """One bandit per finding. Arms = candidate abstention thresholds."""

    candidate_thresholds: list[float] = field(default_factory=lambda: [0.55, 0.6, 0.65, 0.7, 0.75, 0.8])
    epsilon: float = 0.15
    seed: int | None = None

    def __post_init__(self):
        self._rng = random.Random(self.seed)
        self.counts = {t: 0 for t in self.candidate_thresholds}
        self.value_estimates = {t: 0.0 for t in self.candidate_thresholds}

    def select_threshold(self) -> float:
        """Epsilon-greedy: explore a random arm with prob epsilon, else exploit the best-known arm."""
        if self._rng.random() < self.epsilon:
            return self._rng.choice(self.candidate_thresholds)
        return max(self.value_estimates, key=self.value_estimates.get)

    def update(self, threshold: float, reward: float) -> None:
        """Incremental sample-average update — standard bandit update rule."""
        self.counts[threshold] += 1
        n = self.counts[threshold]
        old_estimate = self.value_estimates[threshold]
        self.value_estimates[threshold] = old_estimate + (reward - old_estimate) / n

    def best_threshold(self) -> float:
        return max(self.value_estimates, key=self.value_estimates.get)


def compute_reward(model_prob: float, threshold: float, reviewer_accepted: bool) -> float:
    """
    Reward design: abstaining (model_prob < threshold) is "correct" if the
    reviewer would have rejected the prediction anyway (low reviewer trust in
    that call); predicting (model_prob >= threshold) is "correct" if the
    reviewer accepted it. Wrong call in either direction gets penalized.

    reviewer_accepted: ground-truth signal from your accept/reject/edit UI
    (see PROJECT_PLAN.md Phase 4 — the reviewer feedback loop this depends on).
    """
    would_predict = model_prob >= threshold
    if would_predict and reviewer_accepted:
        return 1.0
    if not would_predict and not reviewer_accepted:
        return 1.0  # correctly deferred on a case the reviewer would've rejected
    return -1.0  # either predicted-and-wrong, or abstained-when-reviewer-would-have-accepted


def simulate_feedback_round(bandit: ThresholdBandit, model_prob: float, true_reviewer_accepts: bool) -> float:
    """One bandit training step: pick a threshold, compute reward against the
    (simulated or real) reviewer decision, update the bandit. Returns the reward."""
    threshold = bandit.select_threshold()
    reward = compute_reward(model_prob, threshold, true_reviewer_accepts)
    bandit.update(threshold, reward)
    return reward


## Phase 4 — Demo app

A real Gradio app — `run_pipeline()` is independently testable and was verified to produce DIFFERENT
outputs for different input images (the previous version of this notebook's demo.py returned the same
hardcoded findings regardless of the uploaded image — that bug is fixed here).

In [ ]:
%%writefile /content/drive/MyDrive/cxr-sentinel/src/demo.py
"""
CXR Sentinel — Phase 4 demo app.

Every number shown in this UI comes from an actual forward pass through your
trained checkpoint on the actual uploaded image. There is no hardcoded
findings dict here — if you upload two different images you get two
different outputs, because the model actually ran on each one.

Run in Colab:
    from src.demo import build_demo
    demo = build_demo(checkpoint_path="checkpoints/best_model.pt")
    demo.launch(share=True)

If no checkpoint exists yet, build_demo() will still launch using a freshly
initialized (untrained) model, purely so you can confirm the wiring works —
it will print a loud warning, and the findings will be near-random until you
actually train and pass a real checkpoint.
"""

from __future__ import annotations

import warnings

import numpy as np
import torch
from PIL import Image

from src.calibrate import TemperatureScaler
from src.claim_verify import verify_claims
from src.data import DEFAULT_TARGETS, build_transforms
from src.gradcam import GradCAM
from src.model import CXRClassifier
from src.ood import ConvAutoencoder, reconstruction_error
from src.report_draft import claims_to_json, draft_report_templated
from src.temporal import classify_change, compute_heatmap_iou


def load_model(checkpoint_path: str | None, target_names: list[str], device: str):
    model = CXRClassifier(num_targets=len(target_names), pretrained=False).to(device)
    if checkpoint_path is not None:
        try:
            ckpt = torch.load(checkpoint_path, map_location=device)
            model.load_state_dict(ckpt["model_state_dict"])
            target_names = ckpt.get("target_names", target_names)
        except FileNotFoundError:
            warnings.warn(f"No checkpoint at {checkpoint_path} — using an UNTRAINED model. Findings will be near-random.")
    else:
        warnings.warn("No checkpoint provided — using an UNTRAINED model. Findings will be near-random.")
    model.eval()
    return model, target_names


def _preprocess(pil_image: Image.Image, image_size: int = 224) -> torch.Tensor:
    transform = build_transforms(image_size=image_size, train=False)
    return transform(pil_image.convert("RGB"))


def run_pipeline(
    current_pil: Image.Image,
    prior_pil: Image.Image | None,
    model,
    target_names: list[str],
    device: str,
    ood_model: ConvAutoencoder | None = None,
    ood_threshold: float | None = None,
    positive_threshold: float = 0.5,
):
    """The actual, real prediction function — no UI code here, so it's independently testable."""
    current_tensor = _preprocess(current_pil)
    gradcam = GradCAM(model)

    with torch.no_grad():
        logits = model(current_tensor.unsqueeze(0).to(device))
        probs = torch.sigmoid(logits).squeeze(0).cpu().numpy()

    findings = []
    gradcam_maps = {}
    for i, name in enumerate(target_names):
        heatmap = gradcam(current_tensor.unsqueeze(0).to(device), target_index=i)
        gradcam_maps[name] = heatmap
        findings.append({"finding": name, "probability": float(probs[i])})

    # Phase 2 — only runs if a prior image was actually provided
    if prior_pil is not None:
        prior_tensor = _preprocess(prior_pil)
        with torch.no_grad():
            prior_logits = model(prior_tensor.unsqueeze(0).to(device))
            prior_probs = torch.sigmoid(prior_logits).squeeze(0).cpu().numpy()

        for i, name in enumerate(target_names):
            prior_heatmap = gradcam(prior_tensor.unsqueeze(0).to(device), target_index=i)
            iou = compute_heatmap_iou(gradcam_maps[name], prior_heatmap)
            status = classify_change(float(probs[i]), float(prior_probs[i]), iou, positive_threshold)
            findings[i]["status"] = status
            findings[i]["prior_probability"] = float(prior_probs[i])

    # Unsupervised OOD flag — only runs if an OOD model was actually passed in
    ood_flag = None
    if ood_model is not None and ood_threshold is not None:
        error = reconstruction_error(ood_model, current_tensor.unsqueeze(0), device)[0]
        ood_flag = {"reconstruction_error": float(error), "threshold": ood_threshold, "flagged_ood": bool(error > ood_threshold)}

    # Phase 3 — real templated drafting + real verification, not a mock
    claims = draft_report_templated(findings)
    verification = verify_claims(claims, findings)

    return {
        "findings": findings,
        "gradcam_maps": gradcam_maps,
        "ood": ood_flag,
        "report_json": claims_to_json(claims),
        "verification": [
            {"finding": v.finding, "claim_text": v.claim_text, "verdict": v.verdict, "reason": v.reason} for v in verification
        ],
    }


def overlay_heatmap(pil_image: Image.Image, heatmap: np.ndarray) -> np.ndarray:
    """Blends a Grad-CAM heatmap onto the original image for display."""
    import matplotlib.cm as cm

    img = np.array(pil_image.convert("RGB").resize((heatmap.shape[1], heatmap.shape[0]))) / 255.0
    colored = cm.jet(heatmap)[..., :3]
    blended = 0.6 * img + 0.4 * colored
    return (blended * 255).astype(np.uint8)


def build_demo(checkpoint_path: str | None = None, ood_checkpoint_path: str | None = None, device: str | None = None):
    import gradio as gr

    device = device or ("cuda" if torch.cuda.is_available() else "cpu")
    model, target_names = load_model(checkpoint_path, DEFAULT_TARGETS, device)

    ood_model = None
    ood_threshold = None
    if ood_checkpoint_path is not None:
        ood_model = ConvAutoencoder().to(device)
        ckpt = torch.load(ood_checkpoint_path, map_location=device)
        ood_model.load_state_dict(ckpt["model_state_dict"])
        ood_threshold = ckpt["threshold"]
        ood_model.eval()

    def predict(current_img, prior_img):
        if current_img is None:
            return None, {}, "Upload an image first.", []

        result = run_pipeline(current_img, prior_img, model, target_names, device, ood_model, ood_threshold)

        primary_finding = max(result["findings"], key=lambda f: f["probability"])["finding"]
        overlay = overlay_heatmap(current_img, result["gradcam_maps"][primary_finding])

        return overlay, {"findings": result["findings"], "ood": result["ood"]}, result["report_json"], result["verification"]

    with gr.Blocks(title="CXR Sentinel") as demo:
        gr.Markdown("# CXR Sentinel — live pipeline, not a mock")
        gr.Markdown(
            "Every field below comes from a real forward pass through the loaded checkpoint. "
            "Upload a prior study too to get Phase 2 longitudinal comparison."
        )
        with gr.Row():
            with gr.Column():
                current_image = gr.Image(type="pil", label="Current study")
                prior_image = gr.Image(type="pil", label="Prior study (optional)")
                submit_btn = gr.Button("Analyze")
            with gr.Column():
                gradcam_output = gr.Image(label="Grad-CAM (highest-probability finding)")
                findings_output = gr.JSON(label="Structured findings + OOD flag (Phase 1/2/unsupervised)")
                report_output = gr.Code(language="json", label="Drafted report claims (Phase 3a)")
                verification_output = gr.JSON(label="Claim verification (Phase 3b)")

        submit_btn.click(
            fn=predict,
            inputs=[current_image, prior_image],
            outputs=[gradcam_output, findings_output, report_output, verification_output],
        )

    return demo


if __name__ == "__main__":
    demo = build_demo()
    demo.launch()


## Data downloader

Downloads from the Hugging Face mirror of NIH's official release (same files as
nihcc.app.box.com/v/ChestXray-NIHCC — no account needed). Splits **train/val/test by patient** — test
is held out entirely for a genuine unseen-data evaluation at the end.

**Label caveat:** NIH ChestX-ray14 has no native "Lung Opacity" label (that's CheXpert-specific). This
script maps NIH's "Infiltration" to `lung_opacity` as a placeholder — swap in CheXpert Plus once your
access clears for the real label, same CSV schema, no other code changes needed.

In [ ]:
%%writefile /content/drive/MyDrive/cxr-sentinel/scripts/download_nih_chestxray14.py
"""
CXR Sentinel — dataset download + converter (NIH ChestX-ray14).

Downloads directly from the Hugging Face mirror of the official NIH release
(same files as https://nihcc.app.box.com/v/ChestXray-NIHCC — no account, no
credentialing, no login required, per NIH's own usage terms). Then converts
into this repo's common CSV schema (see src/data.py).

The full dataset is 12 zip batches, ~9,300 images / ~3-4 GB each, ~42 GB
total. You almost never need all 12 for Phase 1 development — this script
lets you grab just a few batches for a fast local/VS Code iteration subset,
then pull the rest later (in Colab, with more disk/bandwidth) for a full
training run.

IMPORTANT — label caveat:
NIH ChestX-ray14 does NOT have a native "Lung Opacity" label (that's a
CheXpert-specific finding). This script maps "Infiltration" -> lung_opacity
as an approximation, since NIH labels were NLP-mined from reports and
Infiltration is the closest available concept. Treat Phase 1 lung_opacity
numbers trained on this data as a placeholder to validate the pipeline —
swap in CheXpert Plus or the RSNA Pneumonia Detection Challenge dataset
(which has a radiologist-defined "Lung Opacity" label) once you have it,
without changing any other code.

Usage:
    python -m scripts.download_nih_chestxray14 --num_batches 1
    python -m scripts.download_nih_chestxray14 --num_batches 12   # full dataset
"""

from __future__ import annotations

import argparse
import os
import shutil
import zipfile
from pathlib import Path

import pandas as pd
import requests

REPO = "https://huggingface.co/datasets/alkzar90/NIH-Chest-X-ray-dataset/resolve/main/data"
LABELS_CSV_URL = f"{REPO}/Data_Entry_2017_v2020.csv"
IMAGE_BATCH_URL = REPO + "/images/images_{batch:03d}.zip"

# NIH's 15 original NLP-mined labels (14 findings + "No Finding").
# "Lung Opacity" is NOT one of them — see module docstring.
NIH_TO_PROJECT_LABELS = {
    "Cardiomegaly": "cardiomegaly",
    "Effusion": "pleural_effusion",
    "Infiltration": "lung_opacity",  # approximation, see module docstring
}


def download_file(url: str, dest: Path, chunk_size: int = 1 << 20) -> None:
    if dest.exists():
        print(f"  already have {dest.name}, skipping download")
        return
    dest.parent.mkdir(parents=True, exist_ok=True)
    tmp = dest.with_suffix(dest.suffix + ".part")
    with requests.get(url, stream=True, timeout=60) as r:
        r.raise_for_status()
        total = int(r.headers.get("content-length", 0))
        downloaded = 0
        with open(tmp, "wb") as f:
            for chunk in r.iter_content(chunk_size=chunk_size):
                f.write(chunk)
                downloaded += len(chunk)
                if total:
                    pct = 100 * downloaded / total
                    print(f"\r  {dest.name}: {downloaded / 1e6:.0f}MB / {total / 1e6:.0f}MB ({pct:.0f}%)", end="")
    print()
    tmp.rename(dest)


def download_and_extract_batch(batch_num: int, raw_dir: Path, images_out_dir: Path) -> int:
    """Downloads one images_XXX.zip, extracts pngs into images_out_dir (flat), returns count added."""
    zip_path = raw_dir / f"images_{batch_num:03d}.zip"
    print(f"[batch {batch_num:03d}] downloading...")
    download_file(IMAGE_BATCH_URL.format(batch=batch_num), zip_path)

    extract_dir = raw_dir / f"images_{batch_num:03d}_extracted"
    print(f"[batch {batch_num:03d}] extracting...")
    with zipfile.ZipFile(zip_path) as zf:
        zf.extractall(extract_dir)

    images_out_dir.mkdir(parents=True, exist_ok=True)
    count = 0
    for png_path in extract_dir.rglob("*.png"):
        target = images_out_dir / png_path.name
        if not target.exists():
            shutil.move(str(png_path), str(target))
            count += 1

    shutil.rmtree(extract_dir, ignore_errors=True)
    print(f"[batch {batch_num:03d}] added {count} images")
    return count


def build_manifest(labels_csv_path: Path, images_dir: Path) -> pd.DataFrame:
    """Cross-references downloaded images against the NIH label CSV and maps to this repo's schema."""
    labels_df = pd.read_csv(labels_csv_path)
    labels_df = labels_df.rename(columns={"Image Index": "image_filename", "Patient ID": "patient_id"})

    available = {p.name for p in images_dir.glob("*.png")}
    labels_df = labels_df[labels_df["image_filename"].isin(available)].copy()

    for nih_label, project_label in NIH_TO_PROJECT_LABELS.items():
        labels_df[project_label] = labels_df["Finding Labels"].apply(
            lambda findings, nl=nih_label: int(nl in str(findings).split("|"))
        )

    labels_df["image_path"] = "images/" + labels_df["image_filename"]
    # NIH doesn't release real study dates or study IDs; Image Index is unique
    # per study here (each row is one image/study), so it doubles as study_id.
    labels_df["study_id"] = labels_df["image_filename"].str.replace(".png", "", regex=False)
    labels_df["study_date"] = ""

    keep_cols = ["image_path", "patient_id", "study_id", "study_date", *NIH_TO_PROJECT_LABELS.values()]
    return labels_df[keep_cols].reset_index(drop=True)


def patient_level_split(manifest: pd.DataFrame, val_fraction: float, test_fraction: float, seed: int):
    """
    Splits by patient_id into train/val/test so no patient's images leak across splits.

    `test` is genuinely held out: nothing in the training or evaluation loop
    (including checkpoint selection, which uses val AUROC) ever looks at it
    until you deliberately run evaluate.py against data/test.csv at the end.
    That's what makes it "unseen data" rather than a second validation set.
    """
    patients = manifest["patient_id"].unique()
    shuffled = pd.Series(patients).sample(frac=1.0, random_state=seed).values

    n_val = max(1, int(len(shuffled) * val_fraction))
    n_test = max(1, int(len(shuffled) * test_fraction))

    val_patients = set(shuffled[:n_val])
    test_patients = set(shuffled[n_val:n_val + n_test])
    # everyone else -> train

    val_mask = manifest["patient_id"].isin(val_patients)
    test_mask = manifest["patient_id"].isin(test_patients)
    train_mask = ~val_mask & ~test_mask

    return (
        manifest[train_mask].reset_index(drop=True),
        manifest[val_mask].reset_index(drop=True),
        manifest[test_mask].reset_index(drop=True),
    )


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--num_batches", type=int, default=1, help="How many of the 12 image batches to download (1 batch ~= 9,300 images, ~3-4GB)")
    parser.add_argument("--out_dir", default="data")
    parser.add_argument("--val_fraction", type=float, default=0.15)
    parser.add_argument("--test_fraction", type=float, default=0.15, help="Held out entirely — never used for training or checkpoint selection, only for final evaluation on unseen data.")
    parser.add_argument("--seed", type=int, default=42)
    args = parser.parse_args()

    if not 1 <= args.num_batches <= 12:
        raise ValueError("--num_batches must be between 1 and 12")

    out_dir = Path(args.out_dir)
    raw_dir = out_dir / "_raw"
    images_dir = out_dir / "images"

    print("Downloading label metadata (Data_Entry_2017_v2020.csv, ~9MB)...")
    labels_csv_path = raw_dir / "Data_Entry_2017_v2020.csv"
    download_file(LABELS_CSV_URL, labels_csv_path)

    total_images = 0
    for batch in range(1, args.num_batches + 1):
        total_images += download_and_extract_batch(batch, raw_dir, images_dir)

    print(f"\nDownloaded {total_images} new images. Building manifest...")
    manifest = build_manifest(labels_csv_path, images_dir)
    print(f"Manifest covers {len(manifest)} images across {manifest['patient_id'].nunique()} patients.")

    train_df, val_df, test_df = patient_level_split(manifest, args.val_fraction, args.test_fraction, args.seed)
    train_df.to_csv(out_dir / "train.csv", index=False)
    val_df.to_csv(out_dir / "val.csv", index=False)
    test_df.to_csv(out_dir / "test.csv", index=False)

    print(
        f"\nWrote {out_dir/'train.csv'} ({len(train_df)} rows), {out_dir/'val.csv'} ({len(val_df)} rows), "
        f"{out_dir/'test.csv'} ({len(test_df)} rows, held out — don't touch until final evaluation)."
    )
    print("Positive rates (train):")
    for label in NIH_TO_PROJECT_LABELS.values():
        print(f"  {label}: {train_df[label].mean():.1%}")
    print(
        "\nNote: lung_opacity is approximated from NIH's 'Infiltration' label — see this "
        "script's module docstring before trusting Phase 1 lung_opacity numbers."
    )


if __name__ == "__main__":
    main()


## Full pipeline self-test

Runs Phase 1, Phase 2, the unsupervised OOD model, Phase 3 (including the hallucination-catching
check), and the RL bandit end-to-end on synthetic data. **Run this next, before touching real data.**

In [ ]:
%%writefile /content/drive/MyDrive/cxr-sentinel/src/selftest.py
"""
Runs the ENTIRE pipeline — Phase 1 (data -> train -> Grad-CAM -> calibration),
Phase 2 (temporal/history comparison), unsupervised OOD, Phase 3 (report
drafting + claim verification, including a deliberately hallucinated claim
to prove the verifier catches it), and the RL threshold bandit — on synthetic
data. Proves every module is wired together correctly before you've
downloaded a single real X-ray. Run this first, in Colab or locally.

    python -m src.selftest
"""

from __future__ import annotations

import os
import shutil
import tempfile

import numpy as np
import pandas as pd
import torch
from PIL import Image

from src.calibrate import TemperatureScaler, expected_calibration_error
from src.data import CXRDataset, CXRDatasetConfig, DEFAULT_TARGETS
from src.gradcam import GradCAM
from src.model import CXRClassifier
from src.train import evaluate, run_training


def make_synthetic_dataset(root: str, n_images: int = 40, image_size: int = 320):
    image_dir = os.path.join(root, "images")
    os.makedirs(image_dir, exist_ok=True)

    rng = np.random.default_rng(0)
    rows = []
    for i in range(n_images):
        arr = rng.integers(0, 255, size=(image_size, image_size, 3), dtype=np.uint8)
        fname = f"synthetic_{i:03d}.png"
        Image.fromarray(arr).save(os.path.join(image_dir, fname))

        rows.append(
            {
                "image_path": os.path.join("images", fname),
                "patient_id": f"P{i % 10:04d}",
                "study_id": f"S{i:05d}",
                "study_date": "2026-01-01",
                "cardiomegaly": int(rng.random() < 0.3),
                "pleural_effusion": int(rng.random() < 0.3),
                "lung_opacity": int(rng.random() < 0.3),
            }
        )

    df = pd.DataFrame(rows)
    csv_path = os.path.join(root, "manifest.csv")
    df.to_csv(csv_path, index=False)
    return csv_path, image_dir


def main():
    tmp_root = tempfile.mkdtemp(prefix="cxr_selftest_")
    try:
        csv_path, image_root = make_synthetic_dataset(tmp_root, n_images=40, image_size=224)

        train_cfg = CXRDatasetConfig(csv_path=csv_path, image_root=tmp_root, image_size=224, train=True)
        val_cfg = CXRDatasetConfig(csv_path=csv_path, image_root=tmp_root, image_size=224, train=False)
        train_ds = CXRDataset(train_cfg)
        val_ds = CXRDataset(val_cfg)
        print(f"[ok] datasets built: train={len(train_ds)} val={len(val_ds)}")

        image, labels, meta = train_ds[0]
        print(f"[ok] sample item: image {tuple(image.shape)}, labels {labels.tolist()}, meta keys {list(meta.keys())}")

        output_dir = os.path.join(tmp_root, "checkpoints")
        # pretrained=False here only because this sandbox's network can't reach
        # download.pytorch.org; in Colab (or anywhere with normal internet) leave
        # the default pretrained=True — ImageNet init matters a lot at this data scale.
        model, history = run_training(
            train_ds, val_ds, DEFAULT_TARGETS, output_dir, epochs=2, batch_size=8, num_workers=0, pretrained=False
        )
        print(f"[ok] training ran for {len(history)} epochs, checkpoint saved: "
              f"{os.path.exists(os.path.join(output_dir, 'best_model.pt'))}")

        device = "cuda" if torch.cuda.is_available() else "cpu"
        model.eval()
        sample_image = image.unsqueeze(0).to(device)
        cam = GradCAM(model)
        heatmap = cam(sample_image, target_index=0)
        assert heatmap.shape == (224, 224), heatmap.shape
        print(f"[ok] grad-cam heatmap shape {heatmap.shape}, range [{heatmap.min():.3f}, {heatmap.max():.3f}]")

        import torch.nn as nn
        from torch.utils.data import DataLoader

        val_loader = DataLoader(val_ds, batch_size=8, num_workers=0)
        _, _, val_logits, val_labels = evaluate(model, val_loader, nn.BCEWithLogitsLoss(), device, DEFAULT_TARGETS)

        scaler = TemperatureScaler(num_targets=len(DEFAULT_TARGETS))
        learned_temp = scaler.fit(val_logits, val_labels)
        print(f"[ok] temperature scaling fit, learned temperatures: {learned_temp.tolist()}")

        raw_probs = torch.sigmoid(val_logits)[:, 0].numpy()
        calibrated_probs = torch.sigmoid(scaler(val_logits))[:, 0].detach().numpy()
        labels_np = val_labels[:, 0].numpy()

        ece_raw = expected_calibration_error(raw_probs, labels_np, n_bins=5)
        ece_calibrated = expected_calibration_error(calibrated_probs, labels_np, n_bins=5)
        print(f"[ok] ECE raw={ece_raw:.4f} calibrated={ece_calibrated:.4f}")

        # --- Phase 2: temporal / history comparison ---
        from src.temporal import compare_studies, pair_studies

        toy_manifest = pd.DataFrame({
            "patient_id": ["P1", "P1"], "study_id": ["s1", "s2"], "study_date": ["", ""],
        })
        pairs = pair_studies(toy_manifest)
        assert len(pairs) == 1
        prior_image, _, _ = train_ds[1]
        change_results = compare_studies(model, cam, sample_image.squeeze(0), prior_image, DEFAULT_TARGETS, device=device)
        assert len(change_results) == len(DEFAULT_TARGETS)
        print(f"[ok] Phase 2 temporal comparison ran: {[(r.finding, r.status) for r in change_results]}")

        # --- Unsupervised: OOD autoencoder ---
        from torch.utils.data import DataLoader as _DL

        from src.ood import ConvAutoencoder, fit_ood_threshold, reconstruction_error, train_autoencoder

        ae = ConvAutoencoder()
        ae_loader = _DL(torch.rand(16, 3, 64, 64), batch_size=8)
        ae_history = train_autoencoder(ae, ae_loader, epochs=2, device=device)
        assert len(ae_history) == 2
        errs = reconstruction_error(ae, torch.rand(4, 3, 64, 64), device)
        thresh = fit_ood_threshold(errs, percentile=90)
        print(f"[ok] unsupervised OOD autoencoder trained, threshold={thresh:.4f}")

        # --- Phase 3: report drafting + claim verification (incl. a bad claim) ---
        from src.claim_verify import verify_claims
        from src.report_draft import Claim, draft_report_templated

        toy_findings = [{"finding": "cardiomegaly", "probability": 0.9, "status": "worsening"}]
        toy_claims = draft_report_templated(toy_findings)
        toy_verification = verify_claims(toy_claims, toy_findings)
        assert toy_verification[0].verdict == "SUPPORTED"

        # A finding the model never even assessed — the classic hallucination case
        fake_claim = Claim("pneumothorax", "There is a large pneumothorax.", "right lung", 0.9, None)
        bad_verification = verify_claims([fake_claim], toy_findings)
        assert bad_verification[0].verdict == "UNSUPPORTED", "verifier failed to catch a claim about an unassessed finding"
        print("[ok] Phase 3 report drafting + claim verification correctly flags a hallucinated claim")

        # --- RL: threshold bandit ---
        from src.threshold_bandit import ThresholdBandit, simulate_feedback_round

        bandit = ThresholdBandit(seed=0)
        for _ in range(50):
            simulate_feedback_round(bandit, model_prob=0.9, true_reviewer_accepts=True)
        assert bandit.counts[bandit.select_threshold()] >= 0  # just confirm no crash across many pulls
        print(f"[ok] threshold bandit ran 50 feedback rounds, current best={bandit.best_threshold()}")

        print("\nALL CHECKS PASSED — full pipeline (Phase 1, 2, 3, unsupervised OOD, RL bandit) wired correctly end to end.")
    finally:
        shutil.rmtree(tmp_root, ignore_errors=True)


if __name__ == "__main__":
    main()


### Run the self-test

In [ ]:
!python -m src.selftest

## Step 1 — Download real data

First batch now (fast, ~9,300 images) to validate on real images; increase `--num_batches` up to 12
for the full dataset once you're ready for a full training run.

In [ ]:
!python -m scripts.download_nih_chestxray14 --num_batches 1 \
    --out_dir /content/drive/MyDrive/cxr-sentinel/data

## Step 2 — Train (Phase 1)

In [ ]:
import yaml
from src.data import CXRDataset, CXRDatasetConfig, DEFAULT_TARGETS
from src.train import run_training

data_dir = '/content/drive/MyDrive/cxr-sentinel/data'
train_cfg = CXRDatasetConfig(csv_path=f'{data_dir}/train.csv', image_root=data_dir, image_size=224, train=True)
val_cfg = CXRDatasetConfig(csv_path=f'{data_dir}/val.csv', image_root=data_dir, image_size=224, train=False)

train_ds = CXRDataset(train_cfg)
val_ds = CXRDataset(val_cfg)
print(f"train={len(train_ds)} val={len(val_ds)}")

model, history = run_training(
    train_ds, val_ds, DEFAULT_TARGETS,
    output_dir=f'/content/drive/MyDrive/cxr-sentinel/checkpoints',
    epochs=15, batch_size=32, lr=1e-4,
)

## Step 3 — Evaluate on validation, THEN on the held-out test set

Run validation first to sanity-check. Only look at `test.csv` once you're actually done iterating —
that's what keeps it a genuine unseen-data number instead of a set you accidentally tuned against.

In [ ]:
!python -m src.evaluate \
    --checkpoint /content/drive/MyDrive/cxr-sentinel/checkpoints/best_model.pt \
    --csv /content/drive/MyDrive/cxr-sentinel/data/val.csv --image_root /content/drive/MyDrive/cxr-sentinel/data \
    --out_dir /content/drive/MyDrive/cxr-sentinel/reports/val

In [ ]:
# Only run this once you're done iterating on val:
!python -m src.evaluate \
    --checkpoint /content/drive/MyDrive/cxr-sentinel/checkpoints/best_model.pt \
    --csv /content/drive/MyDrive/cxr-sentinel/data/test.csv --image_root /content/drive/MyDrive/cxr-sentinel/data \
    --out_dir /content/drive/MyDrive/cxr-sentinel/reports/test_UNSEEN

## Step 4 — Phase 2: longitudinal comparison on real patient pairs

In [ ]:
import pandas as pd
import torch
from src.temporal import pair_studies, compare_studies
from src.gradcam import GradCAM
from src.data import build_transforms
from PIL import Image

manifest = pd.concat([pd.read_csv(f'{data_dir}/train.csv'), pd.read_csv(f'{data_dir}/val.csv')])
pairs = pair_studies(manifest)
print(f"Found {len(pairs)} patients with repeat studies in this batch.")

if pairs:
    p = pairs[0]
    transform = build_transforms(image_size=224, train=False)
    current_img = transform(Image.open(f"{data_dir}/{p['current']['image_path']}").convert('RGB'))
    prior_img = transform(Image.open(f"{data_dir}/{p['prior']['image_path']}").convert('RGB'))

    cam = GradCAM(model)
    results = compare_studies(model, cam, current_img, prior_img, DEFAULT_TARGETS, device='cuda' if torch.cuda.is_available() else 'cpu')
    for r in results:
        print(f"  {r.finding}: {r.prob_prior:.3f} -> {r.prob_current:.3f}  ({r.status})")
else:
    print("No repeat-study patients in this 1-batch subset yet — download more batches, or wait for CheXpert Plus/MIMIC-CXR access, which have much denser repeat-study coverage.")

## Step 5 — Unsupervised: train the OOD autoencoder on normal images

In [ ]:
from torch.utils.data import DataLoader
from src.ood import ConvAutoencoder, train_autoencoder, reconstruction_error, fit_ood_threshold

normal_df = pd.read_csv(f'{data_dir}/train.csv')
normal_df = normal_df[(normal_df['cardiomegaly']==0) & (normal_df['pleural_effusion']==0) & (normal_df['lung_opacity']==0)]

# The autoencoder must only ever see normal images in training — write the filtered
# rows to their own CSV and reuse CXRDataset as-is rather than touching its labels.
normal_csv_path = f'{data_dir}/_normal_only.csv'
normal_df.to_csv(normal_csv_path, index=False)
normal_cfg = CXRDatasetConfig(csv_path=normal_csv_path, image_root=data_dir, image_size=64, train=False)
normal_ds = CXRDataset(normal_cfg)
normal_loader = DataLoader(normal_ds, batch_size=16, shuffle=True)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
ood_model = ConvAutoencoder().to(device)
# normal_loader yields (image_batch, label_batch, meta) per the CXRDataset __getitem__ contract;
# train_autoencoder only uses the image_batch (batch[0]) — labels are never touched, which is what
# keeps this genuinely unsupervised.
history = train_autoencoder(ood_model, normal_loader, epochs=10, device=device)

calib_errors = reconstruction_error(ood_model, next(iter(normal_loader))[0], device)
threshold = fit_ood_threshold(calib_errors, percentile=95)
torch.save({'model_state_dict': ood_model.state_dict(), 'threshold': threshold}, f'/content/drive/MyDrive/cxr-sentinel/checkpoints/ood_model.pt')
print(f"OOD threshold set at {threshold:.5f}, checkpoint saved.")

## Step 6 — Phase 3: draft + verify a report from real predictions

In [ ]:
from src.report_draft import draft_report_templated, claims_to_json
from src.claim_verify import verify_claims

sample_image, sample_labels, meta = val_ds[0]
with torch.no_grad():
    probs = torch.sigmoid(model(sample_image.unsqueeze(0).to(device))).squeeze(0).cpu().numpy()

findings = [{'finding': n, 'probability': float(p)} for n, p in zip(DEFAULT_TARGETS, probs)]
claims = draft_report_templated(findings)
print(claims_to_json(claims))

verification = verify_claims(claims, findings)
for v in verification:
    print(f"  [{v.verdict}] {v.claim_text}")

## Step 7 — RL: run a few bandit feedback rounds

In [ ]:
from src.threshold_bandit import ThresholdBandit, simulate_feedback_round
import random

bandit = ThresholdBandit(seed=0)

# Replace this with real accept/reject clicks from your Phase 4 reviewer UI —
# this block simulates them so the bandit has something to learn from today.
for _ in range(500):
    p = random.random()
    reviewer_would_accept = p >= 0.7  # stand-in for a real reviewer's judgment
    simulate_feedback_round(bandit, p, reviewer_would_accept)

print("Learned abstention threshold:", bandit.best_threshold())
print(bandit.value_estimates)

## Step 8 — Phase 4: launch the real demo app

In [ ]:
from src.demo import build_demo

demo = build_demo(
    checkpoint_path='/content/drive/MyDrive/cxr-sentinel/checkpoints/best_model.pt',
    ood_checkpoint_path='/content/drive/MyDrive/cxr-sentinel/checkpoints/ood_model.pt',
)
demo.launch(share=True)

## What's genuinely out of scope here, and why

- **MedSAM / segmentation** — needs its own pretrained checkpoint and a different training pipeline;
  not something to bolt on without dedicated time.
- **Multi-model consensus** — needs several independently trained specialist models; you have one
  classifier so far.
- **Evidence graph database** — needs a running Postgres/Qdrant instance, which doesn't belong in a
  notebook.
- **Deep ensembles / conformal prediction** — real techniques, meaningfully more compute (multiple
  full training runs) than temperature scaling for a similar calibration benefit at this stage.

None of these are hard, they're just not honest to fake in a single pass. Add them deliberately, one
at a time, the same way Phase 2/3/4 got added here — build it, test it against a known-answer case,
then wire it in.